# Embeddings SPECT y MRI: minmax+KL frente a KL 1e-1

Compara los códigos de los autoencoders de la línea base `minmax_kl` (`KL_LAMBDA=1e-3`) con el experimento reciente `kl1e1` (`KL_LAMBDA=1e-1`). Ambas etapas usan imagen a `[0, 1]` con percentiles fijos de train, KL hacia `N(0, 1)`, decoder `tanh`, y difusión que predice el latente limpio sin z-score.

Las distribuciones usan los 147 sujetos de `data_info/train.txt`, los mismos en las dos etapas y en las dos modalidades. Si un ID de MRI aparece en `train/` y en `test/`, las distribuciones se quedan con la copia de `train/`.

La clasificación aparta un 15 % de ese train como validación. El desempeño que se reporta, y las proyecciones PCA y t-SNE, usan los 34 sujetos de `data_info/test.txt`, cuando el latente de test de esa versión ya está en disco:

- SPECT minmax+KL: `data_rois/latent_Abeta_minmax_kl_test/`
- SPECT kl1e1: `data_rois/latent_Abeta_minmax_kl1e1_test/`
- MRI minmax+KL: `result/exp_nm_s1_mri_rois_reproc_minmax_kl/phase1_mri/embeddings/test/`
- MRI kl1e1: `result/exp_nm_s1_mri_rois_reproc_minmax_kl1e1/phase1_mri/embeddings/test/`


## 1. Configuración

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "config.py").is_file():
            return parent
    raise FileNotFoundError("No se encontró la raíz del repo (config.py)")


ROOT = find_repo_root()
TRAIN_IDS = set((ROOT / "data_info" / "train.txt").read_text().split())
LABELS = pd.read_csv(ROOT / "data_info" / "data_info.csv", dtype={"ID": str})
LABELS["label"] = LABELS["label"].astype(int)
LABEL_NAMES = {0: "control", 1: "parkinson"}
RANDOM_STATE = 42
HIST_RANGE = (-12.0, 12.0)
PLOT_SAMPLE = 150_000
QQ_SAMPLE = 20_000
RNG = np.random.default_rng(0)

STAGES = [
    {
        "key": "minmax_kl",
        "title": "minmax + KL (1e-3)",
        "clamp": 4.0,
        "latent_calibration": False,
        "modalities": {
            "SPECT": ROOT / "data_rois/latent_Abeta_minmax_kl",
            "MRI": ROOT / "result/exp_nm_s1_mri_rois_reproc_minmax_kl/phase1_mri/embeddings",
        },
    },
    {
        "key": "kl1e1",
        "title": "minmax + KL (1e-1)",
        "clamp": 4.0,
        "latent_calibration": False,
        "modalities": {
            "SPECT": ROOT / "data_rois/latent_Abeta_minmax_kl1e1",
            "MRI": ROOT / "result/exp_nm_s1_mri_rois_reproc_minmax_kl1e1/phase1_mri/embeddings",
        },
    },
]
TEST_IDS = sorted(set((ROOT / "data_info" / "test.txt").read_text().split()))
TEST_ROOTS = {
    ("minmax_kl", "SPECT"): ROOT / "data_rois/latent_Abeta_minmax_kl_test",
    ("minmax_kl", "MRI"): ROOT / "result/exp_nm_s1_mri_rois_reproc_minmax_kl/phase1_mri/embeddings",
    ("kl1e1", "SPECT"): ROOT / "data_rois/latent_Abeta_minmax_kl1e1_test",
    ("kl1e1", "MRI"): ROOT / "result/exp_nm_s1_mri_rois_reproc_minmax_kl1e1/phase1_mri/embeddings",
}
VAL_FRACTION = 0.15
STAGE_COLORS = {"minmax_kl": "#4C78A8", "kl1e1": "#54A24B"}
CLASS_COLORS = {"control": "#4C78A8", "parkinson": "#E45756"}
print(f"Train IDs: {len(TRAIN_IDS)}")


## 2. Carga

Cada volumen entra una sola vez. El identificador es el nombre del archivo (`3102.nii.gz` → `3102`). Solo se conservan IDs de train. Si el mismo ID está en `train/` y en `test/`, se usa el de `train/`.

In [ ]:
def latent_id(path: Path) -> str:
    name = path.name
    if name.endswith(".nii.gz"):
        return name[:-7]
    return path.stem


def iter_latent_files(root: Path) -> list[Path]:
    files = []
    for pattern in ("*.nii", "*.nii.gz", "*.npy"):
        files.extend(root.rglob(pattern))
    return sorted(files)


def select_files(root: Path, allowed: set[str], prefer: str = "train") -> dict[str, Path]:
    chosen: dict[str, Path] = {}
    if not root.is_dir():
        return chosen
    for path in iter_latent_files(root):
        subject_id = latent_id(path)
        if subject_id not in allowed:
            continue
        current = chosen.get(subject_id)
        if current is None or path.parent.name == prefer:
            chosen[subject_id] = path
    return chosen


def load_volume(path: Path) -> np.ndarray:
    if path.suffix == ".npy":
        array = np.load(path)
    else:
        array = nib.load(str(path)).get_fdata(dtype=np.float32)
    return np.asarray(array, dtype=np.float32)


def common_train_ids() -> list[str]:
    present = set(TRAIN_IDS)
    for stage in STAGES:
        for root in stage["modalities"].values():
            present &= set(select_files(root, TRAIN_IDS))
    return sorted(present)


COHORT = common_train_ids()
labels_of = LABELS.set_index("ID")["label"].to_dict()
missing_labels = [subject_id for subject_id in COHORT if subject_id not in labels_of]
if missing_labels:
    raise ValueError(f"IDs de train sin etiqueta: {missing_labels[:5]}")
print(f"Cohorte común de train: {len(COHORT)} sujetos")
print(pd.Series([labels_of[i] for i in COHORT]).map(LABEL_NAMES).value_counts().to_dict())


## 3. Distribuciones

**Almacenado** es el código en disco. **Vista difusión** en SPECT coincide con lo almacenado en ambas etapas: la calibración (z-score) está apagada, así que el archivo ya es la entrada a la difusión. La varianza de esa vista es el MSE de predecir la media. El clamp (±4) se dibuja sobre esa vista.

La MRI es la condición. Se compara el código almacenado con `N(0, 1)`, el blanco de la KL. La diferencia entre etapas es el peso de la KL (`1e-3` frente a `1e-1`).

In [ ]:
def summarize(values: np.ndarray, clamp: float) -> dict:
    std = float(values.std())
    absolute = np.abs(values)
    percentiles = {
        name: float(np.percentile(values, pct))
        for name, pct in (("p0.1", 0.1), ("p1", 1.0), ("p50", 50.0), ("p99", 99.0), ("p99.9", 99.9))
    }
    return {
        "n_voxels": int(values.size),
        "mean": float(values.mean()),
        "std": std,
        "variance": float(std * std),
        "min": float(values.min()),
        "max": float(values.max()),
        **percentiles,
        "frac_|z|>1": float(np.mean(absolute > 1.0)),
        "frac_|z|>4": float(np.mean(absolute > 4.0)),
        "frac_fuera_clamp": float(np.mean(absolute > clamp)),
    }


def diffusion_view(values: np.ndarray, calibrate: bool) -> np.ndarray:
    if not calibrate:
        return values
    std = float(values.std())
    if std == 0.0:
        return values - float(values.mean())
    return (values - float(values.mean())) / std


def sample_values(values: np.ndarray, size: int) -> np.ndarray:
    if values.size <= size:
        return values
    return RNG.choice(values, size=size, replace=False)


rows = []
loaded = {}
flat = {}
for stage in STAGES:
    calibrate = bool(stage["latent_calibration"])
    clamp = float(stage["clamp"])
    for modality, root in stage["modalities"].items():
        files = select_files(root, set(COHORT))
        volumes = []
        matrix = []
        for subject_id in COHORT:
            volume = load_volume(files[subject_id]).ravel()
            volumes.append(volume.astype(np.float64))
            matrix.append(volume)
        values = np.concatenate(volumes)
        stored = summarize(values, clamp)
        view = diffusion_view(values, calibrate) if modality == "SPECT" else values
        loaded[(stage["key"], modality)] = {
            "stored": values,
            "view": view,
            "clamp": clamp,
            "title": stage["title"],
        }
        flat[(stage["key"], modality)] = np.stack(matrix).astype(np.float32)
        rows.append({
            "etapa": stage["key"],
            "modalidad": modality,
            "vista": "almacenado",
            "n_sujetos": len(COHORT),
            "clamp": clamp,
            **stored,
        })
        if modality == "SPECT" and calibrate:
            rows.append({
                "etapa": stage["key"],
                "modalidad": modality,
                "vista": "difusion",
                "n_sujetos": len(COHORT),
                "clamp": clamp,
                **summarize(view, clamp),
            })
        print(
            f"{stage['key']} / {modality}: media {stored['mean']:.3f}, "
            f"std {stored['std']:.3f}, var {stored['variance']:.3f}, "
            f"|z|>4 {stored['frac_|z|>4']:.4f}"
        )

distribution = pd.DataFrame(rows)
number_columns = [
    "mean", "std", "variance", "min", "max",
    "p0.1", "p1", "p50", "p99", "p99.9",
    "frac_|z|>1", "frac_|z|>4", "frac_fuera_clamp", "clamp",
]
display(
    distribution.style.format({column: "{:.4f}" for column in number_columns}, na_rep="—")
    .set_caption("147 sujetos de train. La varianza de la vista difusión SPECT es el MSE de predecir la media.")
)

print("Piso del predictor de la media (varianza SPECT de la vista difusión):")
for stage in STAGES:
    block = distribution[(distribution["etapa"] == stage["key"]) & (distribution["modalidad"] == "SPECT")]
    chosen = block[block["vista"] == "difusion"]
    if chosen.empty:
        chosen = block
    print(f"  {stage['key']}: {float(chosen.iloc[0]['variance']):.3f}  (clamp ±{stage['clamp']:g})")


In [ ]:
def density_hist(ax, values, color, label):
    sample = sample_values(values, PLOT_SAMPLE)
    counts, edges = np.histogram(sample, bins=80, range=HIST_RANGE)
    widths = np.diff(edges)
    ax.stairs(counts / (sample.size * widths), edges, color=color, linewidth=1.8, label=label)


def plot_overlay(modality: str, field: str, ylabel: str):
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    xs = np.linspace(*HIST_RANGE, 500)
    for stage in STAGES:
        payload = loaded[(stage["key"], modality)]
        density_hist(ax, payload[field], STAGE_COLORS[stage["key"]], stage["title"])
        color = STAGE_COLORS[stage["key"]]
        ax.axvline(payload["clamp"], color=color, linestyle=":", linewidth=1)
        ax.axvline(-payload["clamp"], color=color, linestyle=":", linewidth=1)
    ax.plot(xs, norm.pdf(xs), color="black", linestyle="--", linewidth=1.2, label="N(0, 1)")
    ax.set_xlim(*HIST_RANGE)
    ax.set_xlabel("valor del vóxel")
    ax.set_ylabel("densidad")
    ax.set_title(f"{modality}: {ylabel}")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()


plot_overlay("SPECT", "stored", "código almacenado")
plot_overlay("SPECT", "view", "vista difusión")
plot_overlay("MRI", "stored", "código almacenado")


In [ ]:
def qq_points(values: np.ndarray):
    sample = np.sort(sample_values(values, QQ_SAMPLE))
    probs = (np.arange(1, sample.size + 1) - 0.5) / sample.size
    return norm.ppf(probs), sample


modalities = ["SPECT", "MRI"]
fig, axes = plt.subplots(len(STAGES), 2, figsize=(10, 3.6 * len(STAGES)), squeeze=False)
for row, stage in enumerate(STAGES):
    for col, modality in enumerate(modalities):
        ax = axes[row, col]
        payload = loaded[(stage["key"], modality)]
        values = payload["view"] if modality == "SPECT" else payload["stored"]
        sample = sample_values(values, PLOT_SAMPLE)
        counts, edges = np.histogram(sample, bins=80, range=HIST_RANGE)
        widths = np.diff(edges)
        ax.stairs(counts / (sample.size * widths), edges, fill=True, color=STAGE_COLORS[stage["key"]], alpha=0.75)
        xs = np.linspace(*HIST_RANGE, 400)
        ax.plot(xs, norm.pdf(xs), color="black", linestyle="--", linewidth=1)
        ax.axvline(payload["clamp"], color="#E45756", linestyle=":", linewidth=1)
        ax.axvline(-payload["clamp"], color="#E45756", linestyle=":", linewidth=1)
        ax.set_xlim(*HIST_RANGE)
        ax.set_title(f"{stage['title']} · {modality}")
        ax.set_xlabel("valor")
fig.suptitle("SPECT en la vista difusión, MRI almacenada", y=1.01)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(len(STAGES), 2, figsize=(10, 3.6 * len(STAGES)), squeeze=False)
for row, stage in enumerate(STAGES):
    for col, modality in enumerate(modalities):
        ax = axes[row, col]
        payload = loaded[(stage["key"], modality)]
        values = payload["view"] if modality == "SPECT" else payload["stored"]
        theoretical, observed = qq_points(values)
        ax.plot(theoretical, observed, ".", color=STAGE_COLORS[stage["key"]], markersize=2, alpha=0.45)
        limit = float(np.percentile(np.abs(theoretical), 99.5))
        ax.plot([-limit, limit], [-limit, limit], color="black", linestyle="--", linewidth=1)
        ax.set_title(f"QQ {stage['title']} · {modality}")
        ax.set_xlabel("cuantil N(0, 1)")
        ax.set_ylabel("cuantil observado")
fig.tight_layout()
plt.show()


## 4. Clasificación

Del train se aparta un 15 % como validación, estratificado por etiqueta y con semilla 42. El ajuste usa el 85 % restante. Ese corte es el mismo para las cuatro combinaciones de etapa (`minmax_kl` / `kl1e1`) y modalidad.

El resultado que se reporta es el de los 34 sujetos de `data_info/test.txt`. Si la carpeta de test de una versión todavía no existe (p. ej. SPECT `kl1e1`), esa fila no se inventa: la celda lo indica y sigue con las que sí están.


In [ ]:
y = np.array([labels_of[subject_id] for subject_id in COHORT], dtype=np.int64)
fit_index, val_index = train_test_split(
    np.arange(len(COHORT)),
    test_size=VAL_FRACTION,
    stratify=y,
    random_state=RANDOM_STATE,
)
print(f"Ajuste: {len(fit_index)}  Validación: {len(val_index)} ({VAL_FRACTION:.0%} del train)")
print("Validación:", pd.Series(y[val_index]).map(LABEL_NAMES).value_counts().to_dict())


def make_models(n_train: int) -> dict:
    knn_auto = min(5, n_train)
    if knn_auto % 2 == 0:
        knn_auto -= 1
    knn_auto = max(1, knn_auto)
    return {
        "KNN_auto": KNeighborsClassifier(n_neighbors=knn_auto, weights="distance"),
        "KNN_K2": KNeighborsClassifier(n_neighbors=min(2, n_train), weights="distance"),
        "SVM_linear": LinearSVC(C=1.0, class_weight="balanced", dual="auto", max_iter=5000, random_state=RANDOM_STATE),
        "LogisticRegression": LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE),
    }


def positive_scores(model, features):
    classifier = model.named_steps["classifier"]
    if isinstance(classifier, LinearSVC):
        return model.decision_function(features)
    return model.predict_proba(features)[:, 1]


def score_split(model, features, labels) -> dict:
    predicted = model.predict(features)
    scores = positive_scores(model, features)
    return {
        "accuracy": accuracy_score(labels, predicted),
        "balanced_accuracy": balanced_accuracy_score(labels, predicted),
        "recall": recall_score(labels, predicted, zero_division=0),
        "f1": f1_score(labels, predicted, zero_division=0),
        "roc_auc": roc_auc_score(labels, scores),
    }


def load_split_matrix(root: Path, subject_ids: list[str], prefer: str):
    files = select_files(root, set(subject_ids), prefer=prefer)
    missing = [subject_id for subject_id in subject_ids if subject_id not in files]
    if missing:
        return None
    matrix = np.stack([load_volume(files[subject_id]).ravel() for subject_id in subject_ids])
    labels = np.array([labels_of[subject_id] for subject_id in subject_ids], dtype=np.int64)
    return matrix.astype(np.float32), labels


METRIC_FORMAT = {
    "accuracy": "{:.3f}",
    "balanced_accuracy": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "roc_auc": "{:.3f}",
}

validation_rows = []
test_rows = []
test_predictions = {}
for stage in STAGES:
    for modality in ("SPECT", "MRI"):
        features = flat[(stage["key"], modality)]
        x_fit, y_fit = features[fit_index], y[fit_index]
        x_val, y_val = features[val_index], y[val_index]
        test_pack = load_split_matrix(TEST_ROOTS[(stage["key"], modality)], TEST_IDS, prefer="test")
        if test_pack is None:
            print(f"{stage['key']} / {modality}: sin test oficial en {TEST_ROOTS[(stage['key'], modality)]}")
        for name, classifier in make_models(len(fit_index)).items():
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", classifier),
            ])
            model.fit(x_fit, y_fit)
            validation_rows.append({
                "etapa": stage["key"],
                "modalidad": modality,
                "modelo": name,
                "conjunto": "validacion",
                **score_split(model, x_val, y_val),
            })
            if test_pack is not None:
                x_test, y_test = test_pack
                predicted = model.predict(x_test)
                test_predictions[(stage["key"], modality, name)] = (
                    np.asarray(y_test),
                    np.asarray(predicted),
                )
                test_rows.append({
                    "etapa": stage["key"],
                    "modalidad": modality,
                    "modelo": name,
                    "conjunto": "test",
                    **score_split(model, x_test, y_test),
                })

validation = pd.DataFrame(validation_rows)
display(
    validation.style.format(METRIC_FORMAT).set_caption(
        f"Validación: {VAL_FRACTION:.0%} del train ({len(val_index)} sujetos). No es el número final."
    )
)

if test_rows:
    official_test = pd.DataFrame(test_rows)
    print(f"Test oficial: {len(TEST_IDS)} sujetos de data_info/test.txt")
    display(
        official_test.style.format(METRIC_FORMAT).set_caption(
            "Desempeño en el test oficial. El modelo se ajustó solo con el 85 % del train."
        )
    )
    pivot = official_test.pivot_table(
        index="modelo",
        columns=["modalidad", "etapa"],
        values="balanced_accuracy",
        aggfunc="first",
    )
    display(
        pivot.style.format("{:.3f}").background_gradient(cmap="Greens", axis=None).set_caption(
            "Balanced accuracy en el test oficial"
        )
    )
else:
    print("Todavía no hay latentes de test oficial para reportar.")


## 4.1 Matrices de confusión en el test oficial

Para cada etapa y modalidad se elige el clasificador con mayor balanced accuracy en `data_info/test.txt`. Las filas son la etiqueta real y las columnas la predicción.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

if not test_rows:
    print("Sin test oficial: no hay matrices de confusión.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(9.2, 8))
    for row, modality in enumerate(("SPECT", "MRI")):
        for col, stage in enumerate(STAGES):
            ax = axes[row, col]
            block = official_test[
                (official_test["etapa"] == stage["key"]) & (official_test["modalidad"] == modality)
            ]
            if block.empty:
                ax.text(0.5, 0.5, "sin test oficial", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                ax.set_title(f"{stage['title']} · {modality}")
                continue
            best = block.sort_values("balanced_accuracy", ascending=False).iloc[0]
            #best = block.sort_values("recall", ascending=False).iloc[0]
            y_true, y_pred = test_predictions[(stage["key"], modality, best["modelo"])]
            matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
            ConfusionMatrixDisplay(matrix, display_labels=["control", "parkinson"]).plot(
                ax=ax, colorbar=False, cmap="Blues"
            )
            ax.set_title(
                f"{stage['title']} · {modality}\n"
                f"{best['modelo']}  BA={best['balanced_accuracy']:.3f}"
                #f"{best['modelo']}  RE={best['recall']:.3f}"
            )
    fig.tight_layout()
    plt.show()


## 5. PCA y t-SNE en 2D

Cada panel proyecta el código almacenado de los 34 sujetos de `data_info/test.txt`, el mismo conjunto en el que se reporta la clasificación. El color es la etiqueta verdadera. t-SNE usa perplejidad 8, adecuada a ese tamaño, y la misma semilla en los paneles `minmax_kl` y `kl1e1`. Si falta el latente de test de una versión, ese panel queda vacío.


In [ ]:
def project_2d(features: np.ndarray, method: str) -> np.ndarray:
    scaled = StandardScaler().fit_transform(features)
    perplexity = min(8, features.shape[0] - 1)
    if method == "pca":
        return PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(scaled)
    return TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=RANDOM_STATE,
    ).fit_transform(scaled)


def scatter_by_class(ax, embedding, labels, title):
    frame = pd.DataFrame(embedding, columns=["x", "y"])
    frame["clase"] = [LABEL_NAMES[int(label)] for label in labels]
    for class_name, color in CLASS_COLORS.items():
        block = frame[frame["clase"] == class_name]
        ax.scatter(block["x"], block["y"], s=36, alpha=0.9, c=color, label=class_name, linewidths=0)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


for method, heading in (("pca", "PCA"), ("tsne", "t-SNE")):
    fig, axes = plt.subplots(2, 2, figsize=(9.5, 8))
    for row, modality in enumerate(("SPECT", "MRI")):
        for col, stage in enumerate(STAGES):
            ax = axes[row, col]
            test_pack = load_split_matrix(
                TEST_ROOTS[(stage["key"], modality)], TEST_IDS, prefer="test"
            )
            title = f"{heading} · {modality} · {stage['title']}"
            if test_pack is None:
                ax.text(0.5, 0.5, "sin test oficial", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                ax.set_title(title)
                continue
            features, test_labels = test_pack
            scatter_by_class(ax, project_2d(features, method), test_labels, title)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()
